In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("../data/cancer_data.csv")
X = data.copy()
X = X.dropna(axis=1, thresh=int(0.5*len(X)))
X.dropna(subset=['Role in Cancer'], inplace=True)
y = X.pop("Role in Cancer")


In [3]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.over_sampling import SMOTE

# Load data
data = pd.read_csv("../data/cancer_data.csv")

# Copy data
X = data.copy()

# Restore dropped columns and handle missing values
columns_to_restore = {
    'Germline': 'no',
    'Hallmark': 'no',
    'Other Germline Mut': 'no',
    'Tier': 0,
    'Somatic': 'no'
}

for col, replacement in columns_to_restore.items():
    if col not in X.columns:  # Add column if missing
        X[col] = replacement
    else:  # Replace missing values
        X.fillna({col:replacement}, inplace=True)

# Remove specific columns from X
columns_to_remove = [
    'Cancer Syndrome',
    'Other Syndrome',
    'Translocation Partner',
    'Tumour Types(Germline)',
    'Gene Symbol',
    'Entrez GeneId',
    'Tumour Types(Somatic)',
]
X.drop(columns=[col for col in columns_to_remove if col in X.columns], inplace=True)

# Drop rows without the target variable
X.dropna(subset=['Role in Cancer'], inplace=True)

# Separate target variable
y = X.pop("Role in Cancer")


# Split data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Encode categorical variables
one_hot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train = one_hot.fit_transform(X_train)
X_test = one_hot.transform(X_test)

# Normalize numerical variables
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Handle class imbalance
smote = SMOTE()
X_train, y_train = smote.fit_resample(X_train, y_train)


C:\Users\anish\AppData\Roaming\Python\Python313\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
C:\Users\anish\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\_tags.py:354: FutureWarning: The SMOTE or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


In [4]:
X.columns

Index(['Name', 'Genome Location', 'Tier', 'Hallmark', 'Chr Band', 'Somatic',
       'Germline', 'Tissue Type', 'Molecular Genetics', 'Mutation Types',
       'Other Germline Mut', 'Synonyms'],
      dtype='object')

In [5]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
# Encode target labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

In [6]:
end_model = XGBClassifier(colsample_bytree= 0.8,learning_rate= 0.01,max_depth= 7,n_estimators= 600,subsample= 0.8, random_state=42)
end_model.fit(X_train, y_train_encoded)
end_pred = end_model.predict(X_test)
from sklearn.metrics import accuracy_score
print("Final Accuracy:", accuracy_score(y_test_encoded, end_pred))

Final Accuracy: 0.592274678111588


### Used to find the best parameter for XGBCLassifier

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Encode target labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Define parameter grid
param_grid = {
    'n_estimators': [400, 500, 600],
    'learning_rate': [0.01, 0.03, 0.05],
    'max_depth': [5, 7, 9],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}

# Perform Grid Search
grid_search = GridSearchCV(
    XGBClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    verbose=2
)

grid_search.fit(X_train, y_train_encoded)  # Use encoded labels
best_model = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)

# If you need to decode predictions later
# decoded_predictions = label_encoder.inverse_transform(predictions)


### Best Parameters were - 
Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 600, 'subsample': 0.8}

In [ ]:
final_model = best_model
final_model.fit(X_train, y_train_encoded)
final_pred = final_model.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score
print("Final Accuracy:", accuracy_score(y_test_encoded, final_pred))


In [ ]:
end_model = XGBClassifier(device='cuda',colsample_bytree= 0.8,learning_rate= 0.01,max_depth= 7,n_estimators= 600,subsample= 0.8, random_state=42)
end_model.fit(X_train, y_train_encoded)
end_pred = end_model.predict(X_test)
from sklearn.metrics import accuracy_score
print("Final Accuracy:", accuracy_score(y_test_encoded, end_pred))